# moviepy — Final composite (timeline-anchored)

Composes the polls-tutorial demo into a single mp4 from:

- per-scene narration: `audio/scene_NN.mp3` (committed; rendered by `narrator.ipynb`)
- per-scene Manim clips: `media/videos/source/480p15/<SceneClass>.mp4` (gitignored; rendered by `manim.ipynb`)

**Why timeline placement instead of concatenation?**

`concatenate_videoclips(method="chain")` accumulates each clip's reported `duration` to compute the next start. Manim mp4s have a small frame-count vs duration mismatch (the "bytes wanted but 0 bytes read" warnings during the read pass), and that error compounds across 9 scenes — drift of ~0.3 s by scene 9 was perceptible.

This notebook instead places **every** sub-clip explicitly on the global timeline via `with_start(t)`, where `t` is computed from the **audio** clock (44100 Hz, no rounding error). Video clips can drift internally without affecting audio sync, because each scene's narration is anchored to its scene's start time independently.

`CompositeVideoClip` then renders the timeline frame-by-frame at a uniform 15 fps; whichever sub-clip "owns" a given frame time `t` provides the pixels.

`out/` is gitignored — re-run this notebook to regenerate on a fresh checkout.


In [ ]:
from pathlib import Path

import numpy as np

from moviepy import (
    AudioFileClip,
    CompositeAudioClip,
    CompositeVideoClip,
    ImageClip,
    VideoFileClip,
)

# Notebook lives in source/, so paths are relative to that.
AUDIO_DIR = Path("audio")
VIDEO_DIR = Path("media/videos/source/480p15")
OUT_DIR = Path("out")
OUT_DIR.mkdir(exist_ok=True)
OUT_PATH = OUT_DIR / "polls_demo.mp4"

assert AUDIO_DIR.exists(), f"missing {AUDIO_DIR.resolve()}"
assert VIDEO_DIR.exists(), (
    f"missing {VIDEO_DIR.resolve()} — re-run manim.ipynb to render the scene mp4s"
)

# Output format — match the Manim 480p15 input so we don't resample frames.
FRAME_SIZE = (854, 480)
FPS = 15

print(f"audio dir: {AUDIO_DIR.resolve()}")
print(f"video dir: {VIDEO_DIR.resolve()}")
print(f"out      : {OUT_PATH.resolve()}")


In [ ]:
# Per-scene assets. Order matches the conte.md storyboard.
SCENES = [
    {"id": 1, "audio": "scene_01.mp3", "videos": ["Scene1Opening.mp4"]},
    {"id": 2, "audio": "scene_02.mp3", "videos": ["Scene2Reinhardt.mp4"]},
    {"id": 3, "audio": "scene_03.mp3", "videos": ["Scene3aTree.mp4", "Scene3bMakeTasks.mp4"]},
    {"id": 4, "audio": "scene_04.mp3", "videos": ["Scene4aModelMacro.mp4", "Scene4bQuestionChoiceERD.mp4"]},
    {"id": 5, "audio": "scene_05.mp3", "videos": ["Scene5aServerFnRPC.mp4", "Scene5bUseActionCycle.mp4"]},
    {"id": 6, "audio": "scene_06.mp3", "videos": ["Scene6aFormMacro.mp4"]},
    {"id": 7, "audio": "scene_07.mp3", "videos": ["Scene7aAdminRegister.mp4"]},
    {"id": 8, "audio": "scene_08.mp3", "videos": ["Scene8aDIProviders.mp4", "Scene8bGuardChain.mp4"]},
    {"id": 9, "audio": "scene_09.mp3", "videos": ["Scene9Closing.mp4"]},
]

# Tail silence appended to each scene. The audio clock advances by
# (audio.duration + TAIL_SILENCE), so the next scene starts after a brief beat.
TAIL_SILENCE = 0.6  # seconds

# Audio gain to compensate for moviepy + ffmpeg's mono->stereo AAC RMS attenuation.
AUDIO_GAIN = 1.5  # ~+3.5 dB


In [ ]:
def build_scene_segments(scene, scene_start):
    """Return (video_segments_for_scene, audio_segment, scene_target_duration).

    Each video segment is anchored to the global timeline via with_start.
    Sub-clips of the scene play sequentially; if their total is shorter than
    the audio + tail, a freeze of the last frame fills the remainder.
    """
    audio = AudioFileClip(str(AUDIO_DIR / scene["audio"]))
    target = audio.duration + TAIL_SILENCE

    video_segments = []
    sub_cursor = 0.0  # offset within the scene
    last_video_for_freeze = None

    for vname in scene["videos"]:
        vc = VideoFileClip(str(VIDEO_DIR / vname)).without_audio()
        # Anchor on the global timeline.
        video_segments.append(vc.with_start(scene_start + sub_cursor))
        sub_cursor += vc.duration
        last_video_for_freeze = vc

    if sub_cursor + 1e-3 < target:
        # Hold the last frame to fill the remainder of this scene.
        # Pull the frame from a slightly-before-end timestamp so we don't
        # hit Manim's "frame N+1 missing" edge case.
        freeze_frame = last_video_for_freeze.get_frame(
            max(0.0, last_video_for_freeze.duration - 0.05)
        )
        freeze = (
            ImageClip(freeze_frame)
            .with_duration(target - sub_cursor)
            .with_start(scene_start + sub_cursor)
        )
        freeze.fps = FPS
        video_segments.append(freeze)
    elif sub_cursor > target:
        # Trim the last video to fit the scene's target duration.
        # This branch is unused with the current narration since every
        # scene's audio outlasts its Manim concat; left in for future safety.
        excess = sub_cursor - target
        trimmed = last_video_for_freeze.subclipped(
            0, last_video_for_freeze.duration - excess
        )
        # Replace the last segment with the trimmed one (same start time).
        last_start = video_segments[-1].start
        video_segments[-1] = trimmed.with_start(last_start)

    audio_segment = audio.with_start(scene_start)
    return video_segments, audio_segment, target


# Pre-flight duration plan, computed from audio (the canonical clock).
print(f"{'SCENE':>5} {'AUDIO':>10} {'MANIM':>10} {'OUT':>10} {'CUMUL':>10}")
total = 0.0
for scene in SCENES:
    a = AudioFileClip(str(AUDIO_DIR / scene["audio"])).duration
    v = sum(VideoFileClip(str(VIDEO_DIR / vn)).duration for vn in scene["videos"])
    out = a + TAIL_SILENCE
    total += out
    print(f"   {scene['id']:>2}  {a:8.2f}s  {v:8.2f}s  {out:8.2f}s  {total:8.2f}s")
print(f"TOTAL: {total:.2f}s  ({total / 60:.2f} min)")


In [ ]:
# Build all video and audio segments, anchored to the global timeline.
all_video_segments = []
all_audio_segments = []
cursor = 0.0

for scene in SCENES:
    vsegs, aseg, target = build_scene_segments(scene, cursor)
    all_video_segments.extend(vsegs)
    all_audio_segments.append(aseg)
    cursor += target

TOTAL_DURATION = cursor

# Compose video as a single timeline. CompositeVideoClip renders frame-by-frame:
# at each frame time t, it picks the segment whose [start, start+duration] covers t.
final_video = CompositeVideoClip(
    all_video_segments, size=FRAME_SIZE, bg_color=(26, 27, 38)  # Tokyo Night BG
).with_duration(TOTAL_DURATION)

# Compose audio similarly. Gaps between segments produce natural silence.
final_audio = (
    CompositeAudioClip(all_audio_segments)
    .with_volume_scaled(AUDIO_GAIN)
    .with_duration(TOTAL_DURATION)
)
final_audio.fps = 44100

# Attach.
final = final_video.with_audio(final_audio)

print(f"video segments: {len(all_video_segments)}")
print(f"audio segments: {len(all_audio_segments)}")
print(f"total duration: {TOTAL_DURATION:.3f}s ({TOTAL_DURATION / 60:.2f} min)")
print(f"audio gain    : {AUDIO_GAIN}× (~{20 * np.log10(AUDIO_GAIN):.1f} dB)")


In [ ]:
print(f"Writing {OUT_PATH} ...")

final.write_videofile(
    str(OUT_PATH),
    fps=FPS,
    codec="libx264",
    audio_codec="aac",
    audio_bitrate="192k",
    audio_fps=44100,
    threads=4,
    preset="medium",
    # Force constant-frame-rate output. -vsync cfr makes ffmpeg pad/drop
    # frames to the exact target fps timeline, eliminating any residual
    # drift from inputs that report slightly different fps.
    ffmpeg_params=["-vsync", "cfr"],
)

print("Done.")


In [ ]:
# Sanity check the final mp4.
clip = VideoFileClip(str(OUT_PATH))
size_mb = OUT_PATH.stat().st_size / 1024 / 1024
print(f"path      : {OUT_PATH}")
print(f"size      : {size_mb:.1f} MB")
print(f"duration  : {clip.duration:.3f}s ({clip.duration / 60:.2f} min)")
print(f"fps       : {clip.fps}")
print(f"resolution: {clip.size}")
print(f"audio?    : {clip.audio is not None}")
if clip.audio is not None:
    print(f"audio fps : {clip.audio.fps}")
    print(f"audio dur : {clip.audio.duration:.3f}s")
    drift = clip.duration - clip.audio.duration
    print(f"a/v drift : {drift * 1000:+.1f} ms")
clip.close()
